Creating the Dataset for the ML model by cleaning the existing nuclear powerplant dataset, 2000 random geographic coordinates were generated and added as negative samples. These negative points represent locations where nuclear power plants do not exist, helping the model learn the difference between positive and negative instances during training.

In [76]:
import pandas as pd 
import geopandas as gpd
import random
from shapely.geometry import Point
import geopandas as gpd
import geodatasets

In [77]:
df = pd.read_csv("/Users/mohammadbilal/Documents/Projects/N-Project/data/processed/positive_sites.csv")
print("Before cleaning:")
print(df.info())

Before cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 470 entries, 0 to 469
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Id                   470 non-null    int64  
 1   Name                 470 non-null    object 
 2   Latitude             470 non-null    float64
 3   Longitude            470 non-null    float64
 4   Country              470 non-null    object 
 5   CountryCode          470 non-null    object 
 6   Status               470 non-null    object 
 7   ReactorType          470 non-null    object 
 8   ReactorModel         470 non-null    object 
 9   ConstructionStartAt  470 non-null    object 
 10  OperationalFrom      405 non-null    object 
 11  OperationalTo        0 non-null      float64
 12  Capacity             470 non-null    float64
 13  LastUpdatedAt        470 non-null    object 
 14  Source               470 non-null    object 
 15  IAEAId               46

In [78]:
columns_to_drop = ["Id", "Name", "CountryCode", "Status", "ReactorType", "ReactorModel", "ConstructionStartAt",
                   "OperationalFrom", "OperationalTo", "Capacity", "LastUpdatedAt", "Source", "IAEAId"]
df_cleaned = df.drop(columns=columns_to_drop)
df_cleaned["PowerPlant Site"] = True # Add a column to indicate these are power plant sites
print("\nAfter cleaning:")
print(df_cleaned.head())


After cleaning:
    Latitude  Longitude Country  PowerPlant Site
0  69.709579  170.30625  Russia             True
1  69.709579  170.30625  Russia             True
2  39.807000   -5.69800   Spain             True
3  39.807000   -5.69800   Spain             True
4 -23.008000  -44.45700  Brazil             True


In [79]:
# Adding random negative samples (non-nuclear sites) for balance
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)
world = world[world["CONTINENT"] != "Antarctica"]
# Function to generate random points within a polygon
def random_points_in_polygon(polygon, num_points):
    minx, miny, maxx, maxy = polygon.bounds
    points = []

    while len(points) < num_points:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if polygon.contains(p):
            points.append(p)

    return points

In [80]:
import random
from shapely.geometry import Point

def random_points_in_polygon(poly, n):
    minx, miny, maxx, maxy = poly.bounds
    points = []

    while len(points) < n:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if poly.contains(p):
            points.append(p)

    return points

In [81]:
data = []

for _, row in world.iterrows():
    country = row["ADMIN"]
    polygon = row.geometry
    
    points = random_points_in_polygon(polygon, 20)

    for p in points:
        data.append({
            "Latitude": p.y,
            "Longitude": p.x,
            "Country": country,
        })

In [82]:
random_df = pd.DataFrame(data)
random_df["PowerPlant Site"] = False # Add a column to indicate these are non-power plant sites

In [83]:
print(random_df.head())

    Latitude   Longitude Country  PowerPlant Site
0 -17.975498  177.691570    Fiji            False
1 -18.137660  178.261243    Fiji            False
2 -17.888804  177.734648    Fiji            False
3 -17.534493  178.547844    Fiji            False
4 -16.442185  179.513486    Fiji            False


In [ ]:
df_merged = pd.concat([df_cleaned, random_df], ignore_index=True)
df_merged.to_csv("/Users/mohammadbilal/Documents/Projects/N-Project/output/combined_sites.csv", index=False)

In [86]:
gdf = gpd.GeoDataFrame(
    df_merged,
    geometry=gpd.points_from_xy(df_merged.Longitude, df_merged.Latitude),
    crs="EPSG:4326"
)
gdf.to_file("../output/combined_sites.geojson", driver="GeoJSON")